# User Spec Stats

Run `1_build_user_specs.py` first if `case_specs.json` is missing or stale.

In [1]:
from collections import Counter
from pathlib import Path
import json
import pandas as pd

case_specs_path = Path('data/benchmark/case_specs.json') if Path('data/benchmark').exists() else Path('../benchmark/case_specs.json')
specs = json.loads(case_specs_path.read_text())
len(specs)

300

In [2]:
rows = []
for spec in specs:
    req = spec['user_requirements']
    diag = spec['matching_diagnostics']
    rows.append({
        'case_id': spec['case_id'],
        'difficulty': spec['difficulty'],
        'category': spec['need_category'],
        'county': spec['location']['county'],
        'city': spec['location']['city'],
        'zipcode': spec['location']['zipcode'],
        'location_req': next(k for k in ['counties', 'cities', 'zipcodes'] if k in req),
        'schedule_req': next((k for k in ['available_days', 'available_time_windows', 'requires_24_hours'] if k in req), None),
        'intake_method': req['intake_methods'][0],
        'documents_available_count': len(spec['user_qualification'].get('documents_available', [])),
        'requirements_only_match_count': diag['requirements_only_match_count'],
        'qualified_match_count': diag['qualified_match_count'],
    })

df = pd.DataFrame(rows)
df.head()

,case_id,difficulty,category,county,city,zipcode,location_req,schedule_req,intake_method,documents_available_count,requirements_only_match_count,qualified_match_count
0,spec-001,medium,Food,DELAWARE,Muncie,47305,cities,available_days,walk_in,0,1,1
1,spec-002,easy,Family and Caregiver Support,MARION,Indianapolis,46235,zipcodes,NaN,appointment,2,1,1
2,spec-003,hard,Utility Bill Help,MARION,Indianapolis,46250,counties,available_days,call,0,4,1
3,spec-004,easy,Disaster and Environmental Help,JOHNSON,Franklin,46131,counties,NaN,call,0,1,1
4,spec-005,easy,Material Goods,DECATUR,Greensburg,47240,counties,NaN,walk_in,0,1,1


In [3]:
df['difficulty'].value_counts().rename_axis('difficulty').reset_index(name='count')

,difficulty,count
0,medium,100
1,easy,100
2,hard,100


In [4]:
pd.crosstab(df['difficulty'], df['location_req'])

location_req,cities,counties,zipcodes
difficulty,,,
easy,31,19,50
hard,0,100,0
medium,30,10,60


In [5]:
pd.crosstab(df['difficulty'], df['schedule_req'].fillna('none'))

schedule_req,available_days,available_time_windows,none,requires_24_hours
difficulty,,,,
easy,0,0,100,0
hard,45,54,0,1
medium,45,51,0,4


In [6]:
df.groupby('difficulty')[['requirements_only_match_count', 'qualified_match_count', 'documents_available_count']].agg(['min', 'mean', 'max'])

requirements_only_match_count           qualified_match_count       \
                                     min  mean max                   min mean   
difficulty                                                                      
easy                                   1  1.00   1                     1  1.0   
hard                                   2  2.69  13                     1  1.0   
medium                                 1  1.00   1                     1  1.0   

               documents_available_count            
           max                       min  mean max  
difficulty                                          
easy         1                         0  0.68   6  
hard         1                         0  0.54   4  
medium       1                         0  0.84   6

In [7]:
def explode_req(field):
    values = []
    for spec in specs:
        value = spec['user_requirements'].get(field)
        if isinstance(value, list):
            values.extend(value)
        elif value is not None:
            values.append(str(value).lower() if isinstance(value, bool) else value)
    return pd.Series(values, name=field).value_counts().rename_axis('value').reset_index(name='count')

requirement_fields = Counter(field for spec in specs for field in spec['user_requirements'] if field != 'service_categories')
pd.Series(requirement_fields).sort_values(ascending=False).to_frame('count')

,count
intake_methods,300
counties,129
zipcodes,110
available_time_windows,105
available_days,90
cities,61
requires_24_hours,5


In [8]:
documents = [doc for spec in specs for doc in spec['user_qualification'].get('documents_available', [])]
pd.Series(documents, name='document').value_counts().rename_axis('document').reset_index(name='count')

,document,count
0,photo_id,94
1,proof_of_address,46
2,utility_bill,20
3,proof_of_income,16
4,insurance_card,9
5,lease,9
6,social_security,8
7,birth_certificate,4


In [9]:
df['category'].value_counts().rename_axis('category').reset_index(name='count').head(40)

,category,count
0,Food,36
1,Information and Referral,27
2,Material Goods,23
3,Utility Bill Help,22
4,Housing and Shelter,19
5,Family and Caregiver Support,16
6,Police and Public Safety,16
7,Substance Use Help,14
8,Pet and Animal Services,14
9,Pregnancy and Reproductive Health,13
